In [ ]:
import os
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Enter your Hugging Face token: ")
login(token=hf_token)

cry end to end

In [ ]:
# Clone the repo
!git clone https://github.com/gveres/donateacry-corpus.git

# Move into repo
%cd donateacry-corpus

In [ ]:
import os

base_path = "/content/donateacry-corpus/donateacry_corpus_cleaned_and_updated_data"

# Supported audio formats
audio_extensions = ('.wav', '.mp3', '.caf', '.flac')

total_files = 0

print("📊 File count per tag:\n")

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        count = 0

        # Count audio files (including nested if any)
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(audio_extensions):
                    count += 1

        total_files += count
        print(f"{folder}: {count}")

print("\n🔢 Total number of audio files:", total_files)

In [ ]:
!pip install librosa tqdm

In [ ]:
!pip install nest_asyncio -q

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()   # Upload kaggle.json here

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d sanmithasadhish/infant-cry-dataset --unzip

In [ ]:
"""
=============================================================================
INFANT CRY ANALYSIS — FULL TRAINING PIPELINE (UPDATED)
=============================================================================
Architecture:
  CNN0 (NEW) — ResNet18: Cry vs Non-cry detector
              Trained on Kaggle infant cry dataset (real cry/non-cry labels)
              Replaces fragile RMS/ZCR heuristic gate

  CNN1       — ResNet18: Hungry vs Not-hungry
              Trained on donateacry-corpus

  CNN2       — ResNet18: Subtype classifier (discomfort/tired/belly_pain/burping)
              Trained on donateacry-corpus non-hungry samples

  MedGemma   — Clinical reasoning over waveform + acoustic features + CNN probs

FLOW:
  Audio → CNN0 (cry?) → if cry → CNN1 (hungry?) → if not → CNN2 (subtype)
                                                          → MedGemma reasoning

SETUP:
  Run each cell top to bottom in Google Colab (T4 GPU recommended)
  Requires: Kaggle API credentials + HuggingFace token for MedGemma
=============================================================================
"""

# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# !pip install librosa tqdm scikit-learn transformers accelerate kaggle

# ============================================================
# CELL 2 — Setup Kaggle credentials and download dataset
# ============================================================
# Option A: Upload your kaggle.json in Colab
# from google.colab import files
# files.upload()   # upload kaggle.json

# Option B: Set directly (replace with your credentials)
# import os
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"]      = "your_api_key"

# Then download:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d sanmithasadhish/infant-cry-dataset -p /content/kaggle_cry --unzip
# !ls /content/kaggle_cry

# ============================================================
# CELL 3 — Explore Kaggle dataset structure
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
from PIL import Image
from tqdm import tqdm

KAGGLE_PATH = "/content/kaggle_cry"   # adjust if unzip created a subfolder

print("📂 Kaggle dataset structure:")
for root, dirs, files in os.walk(KAGGLE_PATH):
    level = root.replace(KAGGLE_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"{indent}  {f}")

# ============================================================
# CELL 4 — Build CNN0 metadata (cry vs non-cry)
# ============================================================
"""
The Kaggle dataset (sanmithasadhish/infant-cry-dataset) typically has:
  - A folder for cry audio
  - A folder for non-cry audio (or labeled in filenames)

Adjust CRY_FOLDER and NON_CRY_FOLDER to match what you see in Cell 3.
Common structures:
  /content/kaggle_cry/cry/       + /content/kaggle_cry/non_cry/
  /content/kaggle_cry/1/         + /content/kaggle_cry/0/
  /content/kaggle_cry/positive/  + /content/kaggle_cry/negative/
"""

# ---- ADJUST THESE PATHS after checking Cell 3 output ----
CRY_FOLDER     = "/content/donateacry-corpus/Dataset/cry"
NON_CRY_FOLDER = "/content/donateacry-corpus/Dataset/not_cry" # folder containing non-cry audio
# ----------------------------------------------------------

audio_extensions = ('.wav', '.mp3', '.ogg', '.flac', '.m4a')

cnn0_metadata = []

for fpath in os.listdir(CRY_FOLDER):
    if fpath.lower().endswith(audio_extensions):
        cnn0_metadata.append({
            "audio_path": os.path.join(CRY_FOLDER, fpath),
            "label": 1,   # 1 = cry
            "label_name": "cry"
        })

for fpath in os.listdir(NON_CRY_FOLDER):
    if fpath.lower().endswith(audio_extensions):
        cnn0_metadata.append({
            "audio_path": os.path.join(NON_CRY_FOLDER, fpath),
            "label": 0,   # 0 = non_cry
            "label_name": "non_cry"
        })

df_cnn0 = pd.DataFrame(cnn0_metadata)
print(f"\nCNN0 dataset:")
print(df_cnn0["label_name"].value_counts())
print(f"Total: {len(df_cnn0)} samples")


# ============================================================
# CELL 5 — Convert CNN0 audio to spectrograms
# ============================================================
CNN0_SPEC_PATH = "/content/spectrograms_cnn0"
os.makedirs(CNN0_SPEC_PATH, exist_ok=True)
IMG_SIZE = 224

def audio_to_spectrogram(audio_path, save_path):
    """Convert audio to normalized mel spectrogram PNG."""
    try:
        y, sr = librosa.load(audio_path, sr=22050, duration=10.0)  # cap at 10s
        mel_spec    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_norm    = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        mel_img     = (mel_norm * 255).astype(np.uint8)
        img = Image.fromarray(mel_img).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        img.save(save_path)
        return True
    except Exception as e:
        print(f"⚠️  Failed {audio_path}: {e}")
        return False


image_paths = []
valid_indices = []

for i, row in tqdm(df_cnn0.iterrows(), total=len(df_cnn0), desc="CNN0 spectrograms"):
    fname    = os.path.splitext(os.path.basename(row["audio_path"]))[0]
    savepath = os.path.join(CNN0_SPEC_PATH, f"{fname}_{row['label']}.png")

    if audio_to_spectrogram(row["audio_path"], savepath):
        image_paths.append(savepath)
        valid_indices.append(i)

df_cnn0 = df_cnn0.loc[valid_indices].copy()
df_cnn0["image_path"] = image_paths
df_cnn0.to_csv("/content/cnn0_metadata.csv", index=False)
print(f"✅ CNN0 spectrograms done: {len(df_cnn0)} valid samples")


# ============================================================
# CELL 6 — PyTorch setup + shared dataset class
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class SpectrogramDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df.iloc[idx]["image_path"]).convert("RGB")
        label = int(self.df.iloc[idx]["label"])
        if self.transform:
            image = self.transform(image)
        return image, label


transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def train_and_evaluate(model, train_loader, val_loader, optimizer, criterion,
                       epochs=10, label="Model", scheduler=None):
    """Generic training loop with per-epoch validation + classification report."""
    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if scheduler:
            scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                preds = torch.argmax(model(images.to(device)), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        correct = sum(p == l for p, l in zip(all_preds, all_labels))
        acc = correct / len(all_labels)

        if acc > best_acc:
            best_acc = acc

        print(f"[{label}] Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.3f}")

    print(f"\n[{label}] Best Val Acc: {best_acc:.3f}")
    print(f"[{label}] Final classification report:")
    print(classification_report(all_labels, all_preds, zero_division=0))


# ============================================================
# CELL 7 — Train CNN0: Cry vs Non-cry detector
# ============================================================
df_cnn0["label"] = df_cnn0["label"].astype(int)

train_c0, val_c0 = train_test_split(
    df_cnn0, test_size=0.2, stratify=df_cnn0["label"], random_state=42
)

# Weighted sampler for class balance
cc = train_c0["label"].value_counts().to_dict()
total_c0 = sum(cc.values())
sw_c0 = torch.DoubleTensor(train_c0["label"].map({k: total_c0/v for k,v in cc.items()}).values)
sampler_c0 = WeightedRandomSampler(sw_c0, len(sw_c0))

train_c0_loader = DataLoader(SpectrogramDataset(train_c0, transform_train), batch_size=16, sampler=sampler_c0)
val_c0_loader   = DataLoader(SpectrogramDataset(val_c0,   transform_val),   batch_size=16, shuffle=False)

# ResNet18 — full fine-tune (we have enough data from Kaggle)
model0 = models.resnet18(pretrained=True)
model0.fc = nn.Linear(model0.fc.in_features, 2)
model0 = model0.to(device)

criterion0 = nn.CrossEntropyLoss()
optimizer0 = optim.Adam(model0.parameters(), lr=1e-4)
scheduler0 = optim.lr_scheduler.StepLR(optimizer0, step_size=5, gamma=0.5)

print("🟣 Training CNN0 — Cry vs Non-cry detector...")
train_and_evaluate(
    model0, train_c0_loader, val_c0_loader,
    optimizer0, criterion0,
    epochs=12, label="CNN0-CryDetector",
    scheduler=scheduler0
)

torch.save(model0.state_dict(), "/content/cnn0_cry_detector.pth")
print("✅ CNN0 saved → /content/cnn0_cry_detector.pth")


# ============================================================
# CELL 8 — Clone donateacry corpus + explore
# ============================================================
# !git clone https://github.com/gveres/donateacry-corpus.git

BASE_AUDIO_PATH = "/content/donateacry-corpus/donateacry_corpus_cleaned_and_updated_data"

print("📊 Donateacry file count per tag:")
for folder in sorted(os.listdir(BASE_AUDIO_PATH)):
    folder_path = os.path.join(BASE_AUDIO_PATH, folder)
    if os.path.isdir(folder_path):
        count = sum(1 for _, _, files in os.walk(folder_path) for f in files if f.endswith('.wav'))
        print(f"  {folder}: {count}")


# ============================================================
# CELL 9 — Convert donateacry audio to spectrograms
# ============================================================
OUTPUT_SPEC_PATH = "/content/spectrograms"
os.makedirs(OUTPUT_SPEC_PATH, exist_ok=True)

metadata = []

for folder in os.listdir(BASE_AUDIO_PATH):
    folder_path = os.path.join(BASE_AUDIO_PATH, folder)
    if not os.path.isdir(folder_path):
        continue

    save_folder = os.path.join(OUTPUT_SPEC_PATH, folder)
    os.makedirs(save_folder, exist_ok=True)

    for file in tqdm(os.listdir(folder_path), desc=f"Processing {folder}"):
        if not file.lower().endswith(".wav"):
            continue

        audio_path = os.path.join(folder_path, file)
        label_code = file.split("-")[-1].replace(".wav", "").strip()
        save_path  = os.path.join(save_folder, file.replace(".wav", ".png"))

        if audio_to_spectrogram(audio_path, save_path):
            metadata.append({
                "audio_path":     audio_path,
                "image_path":     save_path,
                "folder_label":   folder,
                "filename_label": label_code
            })

print(f"\n✅ Donateacry spectrograms done: {len(metadata)} files")


# ============================================================
# CELL 10 — Build donateacry metadata
# ============================================================
df = pd.DataFrame(metadata)

code_map = {"hu": "hungry", "bp": "belly_pain", "bu": "burping", "dc": "discomfort", "ti": "tired"}
df["decoded_label"] = df["filename_label"].map(code_map)
df = df[df["decoded_label"].notna()].copy()
df["binary_label"] = df["decoded_label"].apply(lambda x: "hungry" if x == "hungry" else "not_hungry")

df.to_csv("/content/cry_metadata.csv", index=False)
print(df["decoded_label"].value_counts())


# ============================================================
# CELL 11 — Train CNN1: Hungry vs Not-hungry
# ============================================================
df_binary = df.copy()
df_binary["label"] = df_binary["decoded_label"].apply(lambda x: 1 if x == "hungry" else 0)

train_b, val_b = train_test_split(df_binary, test_size=0.2, stratify=df_binary["label"], random_state=42)

cc_b  = train_b["label"].value_counts().to_dict()
tot_b = sum(cc_b.values())
sw_b  = torch.DoubleTensor(train_b["label"].map({k: tot_b/v for k,v in cc_b.items()}).values)

train_b_loader = DataLoader(SpectrogramDataset(train_b, transform_train), batch_size=16,
                            sampler=WeightedRandomSampler(sw_b, len(sw_b)))
val_b_loader   = DataLoader(SpectrogramDataset(val_b, transform_val), batch_size=16, shuffle=False)

model1 = models.resnet18(pretrained=True)
model1.fc = nn.Linear(model1.fc.in_features, 2)
model1 = model1.to(device)

criterion1 = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.parameters(), lr=1e-4)

print("🔵 Training CNN1 — Hungry vs Not-hungry...")
train_and_evaluate(model1, train_b_loader, val_b_loader, optimizer1, criterion1,
                   epochs=10, label="CNN1-Binary")

torch.save(model1.state_dict(), "/content/cnn1_binary.pth")
print("✅ CNN1 saved → /content/cnn1_binary.pth")


# ============================================================
# CELL 12 — Train CNN2: Non-hungry subtype classifier
# ============================================================
df_sub = df[df["decoded_label"] != "hungry"].copy()
df_sub["decoded_label"] = df_sub["decoded_label"].str.strip().str.lower()
df_sub["label"], class_names = pd.factorize(df_sub["decoded_label"])

np.save("/content/cnn2_class_names.npy", class_names)
print("CNN2 classes:", list(class_names))
print(df_sub["decoded_label"].value_counts())

train_s, val_s = train_test_split(df_sub, test_size=0.2, stratify=df_sub["label"], random_state=42)

cc_s  = train_s["label"].value_counts().to_dict()
tot_s = sum(cc_s.values())
sw_s  = torch.DoubleTensor(train_s["label"].map({k: tot_s/v for k,v in cc_s.items()}).values)

train_s_loader = DataLoader(SpectrogramDataset(train_s, transform_train), batch_size=16,
                            sampler=WeightedRandomSampler(sw_s, len(sw_s)))
val_s_loader   = DataLoader(SpectrogramDataset(val_s, transform_val), batch_size=16, shuffle=False)

# Freeze all except layer4 (tiny dataset)
model2 = models.resnet18(pretrained=True)
for param in model2.parameters():
    param.requires_grad = False
for param in model2.layer4.parameters():
    param.requires_grad = True

num_classes = len(class_names)
model2.fc = nn.Linear(model2.fc.in_features, num_classes)
model2 = model2.to(device)

cw_s = torch.tensor([tot_s / cc_s[i] for i in range(num_classes)], dtype=torch.float32).to(device)
criterion2 = nn.CrossEntropyLoss(weight=cw_s)
optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, model2.parameters()), lr=1e-4)

print("🟠 Training CNN2 — Subtype classifier...")
train_and_evaluate(model2, train_s_loader, val_s_loader, optimizer2, criterion2,
                   epochs=15, label="CNN2-Subtype")

torch.save(model2.state_dict(), "/content/cnn2_subtype.pth")
print("✅ CNN2 saved → /content/cnn2_subtype.pth")


# ============================================================
# CELL 13 — Test CNN0 cry detector on a sample
# ============================================================
def is_cry_cnn(audio_path, model, threshold=0.5):
    """
    Use trained CNN0 to detect if audio is a cry.
    Returns (is_cry: bool, cry_prob: float, non_cry_prob: float)
    Much more robust than heuristic RMS/ZCR gate.
    """
    spec_path = "/content/temp_cnn0_spec.png"
    if not audio_to_spectrogram(audio_path, spec_path):
        return False, 0.0, 1.0

    image  = Image.open(spec_path).convert("RGB")
    tensor = transform_val(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).cpu().numpy()[0]

    non_cry_prob = float(probs[0])
    cry_prob     = float(probs[1])

    return cry_prob > threshold, cry_prob, non_cry_prob


# Quick test on a donateacry sample
test_audio = df.iloc[0]["audio_path"]
detected, cry_p, non_cry_p = is_cry_cnn(test_audio, model0)
print(f"\nCNN0 test on: {os.path.basename(test_audio)}")
print(f"  Cry probability    : {cry_p:.3f}")
print(f"  Non-cry probability: {non_cry_p:.3f}")
print(f"  Detected as cry    : {detected}")


# ============================================================
# CELL 14 — Full hierarchical inference with CNN0 gate
# ============================================================
class_names = np.load("/content/cnn2_class_names.npy", allow_pickle=True)


def hierarchical_predict(audio_path):
    """
    Full 3-stage pipeline:
      Stage 0: CNN0 → is this a cry at all?
      Stage 1: CNN1 → hungry vs not_hungry
      Stage 2: CNN2 → subtype (if not hungry)

    Returns dict with all probabilities + routing info.
    """
    # Stage 0 — Cry gate (CNN0, trained on Kaggle data)
    spec_path = "/content/temp_spec.png"
    audio_to_spectrogram(audio_path, spec_path)
    tensor = transform_val(Image.open(spec_path).convert("RGB")).unsqueeze(0).to(device)

    model0.eval()
    with torch.no_grad():
        p0 = F.softmax(model0(tensor), dim=1).cpu().numpy()[0]

    cry_prob     = float(p0[1])
    non_cry_prob = float(p0[0])

    if cry_prob <= 0.5:
        return {
            "is_cry": False,
            "cry_prob": cry_prob,
            "non_cry_prob": non_cry_prob,
            "prediction": "non_cry",
            "probs": {"non_cry": non_cry_prob, "cry": cry_prob}
        }

    # Stage 1 — Binary (CNN1)
    model1.eval()
    with torch.no_grad():
        p1 = F.softmax(model1(tensor), dim=1).cpu().numpy()[0]

    hungry_p     = float(p1[1])
    not_hungry_p = float(p1[0])

    if hungry_p > 0.5:
        return {
            "is_cry": True,
            "cry_prob": cry_prob,
            "prediction": "hungry",
            "probs": {
                "hungry": hungry_p,
                **{n: 0.0 for n in class_names}
            }
        }

    # Stage 2 — Subtype (CNN2)
    model2.eval()
    with torch.no_grad():
        p2 = F.softmax(model2(tensor), dim=1).cpu().numpy()[0]

    subtype_probs = {class_names[i]: float(p2[i]) * not_hungry_p for i in range(len(class_names))}
    prediction    = max(subtype_probs, key=subtype_probs.get)

    return {
        "is_cry": True,
        "cry_prob": cry_prob,
        "prediction": prediction,
        "probs": {"hungry": hungry_p, **subtype_probs}
    }


# Test it
result = hierarchical_predict(df.iloc[0]["audio_path"])
print("\nFull pipeline test:")
print(f"  Is cry     : {result['is_cry']}")
print(f"  Cry prob   : {result['cry_prob']:.3f}")
print(f"  Prediction : {result['prediction']}")
print(f"  Probs      : {result['probs']}")


# ============================================================
# CELL 15 — Evaluate full pipeline
# ============================================================
from sklearn.metrics import accuracy_score

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
    try:
        res = hierarchical_predict(row["audio_path"])
        results.append({
            "gold": row["decoded_label"],
            "pred": res["prediction"],
            "is_cry": res["is_cry"],
            "cry_prob": res["cry_prob"]
        })
    except Exception as e:
        print(f"⚠️  {e}")

results_df = pd.DataFrame(results)
results_df.to_csv("/content/full_pipeline_results.csv", index=False)

print("\n📊 EVALUATION REPORT")
print(f"Cry detection rate on cry audio : {results_df['is_cry'].mean():.3f}")
print(f"\n5-class report:")
print(classification_report(results_df["gold"], results_df["pred"], zero_division=0))


# ============================================================
# CELL 16 — Summary: what's saved
# ============================================================
print("""
✅ ALL MODELS SAVED:

  /content/cnn0_cry_detector.pth   — CNN0: Cry vs Non-cry (Kaggle-trained)
  /content/cnn1_binary.pth         — CNN1: Hungry vs Not-hungry
  /content/cnn2_subtype.pth        — CNN2: Subtype classifier
  /content/cnn2_class_names.npy    — Class name ordering for CNN2

Now update infant_cry_interface.py:
  CNN0_WEIGHTS = "/content/cnn0_cry_detector.pth"
  And replace is_cry() heuristic with is_cry_cnn() using model0.
""")

CNN3: Skin Infection Classifier

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d shubhamgoel27/dermnet -p /content/dermnet --unzip

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DERMNET_PATH = "/content/dermnet/train"

# ✅ EXACT FOLDER NAMES from your screenshot
TARGET_CLASSES = [
    "Atopic Dermatitis Photos",
    "Cellulitis Impetigo and other Bacterial Infections",
    "Exanthems and Drug Eruptions",
    "Tinea Ringworm Candidiasis and other Fungal Infections",
    "Scabies Lyme Disease and other Infestations and Bites",
    "Warts Molluscum and other Viral Infections",
    "Urticaria Hives"
]

metadata = []

for folder in TARGET_CLASSES:
    folder_path = os.path.join(DERMNET_PATH, folder)
    if os.path.isdir(folder_path):

        for img_file in os.listdir(folder_path):
            if img_file.lower().endswith((".jpg", ".png", ".jpeg")):
                metadata.append({
                    "image_path": os.path.join(folder_path, img_file),
                    "label_name": folder
                })

df_cnn3 = pd.DataFrame(metadata)

# Encode labels
df_cnn3["label"], class_names_cnn3 = pd.factorize(df_cnn3["label_name"])

print("✅ CNN3 Classes:")
for i, name in enumerate(class_names_cnn3):
    print(f"{i} → {name}")

print("\n📊 Samples per class:")
print(df_cnn3["label_name"].value_counts())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------
# Dataset Class
# ----------------------
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df.iloc[idx]["image_path"]).convert("RGB")
        label = int(self.df.iloc[idx]["label"])
        if self.transform:
            image = self.transform(image)
        return image, label

# ----------------------
# Transforms
# ----------------------
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Split
# ----------------------
train_df, val_df = train_test_split(
    df_cnn3,
    test_size=0.2,
    stratify=df_cnn3["label"],
    random_state=42
)

train_loader = DataLoader(SkinDataset(train_df, transform_train), batch_size=16, shuffle=True)
val_loader   = DataLoader(SkinDataset(val_df, transform_val),   batch_size=16, shuffle=False)

# ----------------------
# Model
# ----------------------
model3 = models.resnet18(pretrained=True)
model3.fc = nn.Linear(model3.fc.in_features, len(class_names_cnn3))
model3 = model3.to(device)

# Class weights (important — dataset is imbalanced)
class_counts = train_df["label"].value_counts().sort_index()
total = class_counts.sum()
weights = torch.tensor([total/c for c in class_counts], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model3.parameters(), lr=1e-4)

# ----------------------
# Training Loop
# ----------------------
EPOCHS = 10
best_f1 = 0

for epoch in range(EPOCHS):

    model3.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model3(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- Validation ----
    model3.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model3(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_acc = (np.array(all_preds) == np.array(all_labels)).mean()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {total_loss/len(train_loader):.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"Val F1 (Macro): {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model3.state_dict(), "/content/cnn3_skin_classifier.pth")
        print("✅ Saved best model")

print("\n🔥 Training Complete")

In [ ]:
from sklearn.metrics import classification_report

val_f1_macro = f1_score(all_labels, all_preds, average="macro")
val_f1_weighted = f1_score(all_labels, all_preds, average="weighted")
val_acc = (np.array(all_preds) == np.array(all_labels)).mean()

print(f"\nEpoch {epoch+1}/{EPOCHS}")
print(f"Train Loss: {total_loss/len(train_loader):.4f}")
print(f"Val Acc: {val_acc:.4f}")
print(f"Val F1 (Macro): {val_f1_macro:.4f}")
print(f"Val F1 (Weighted): {val_f1_weighted:.4f}")

# 🔎 Class-wise report
print("\n📊 Class-wise Validation Report:")
report = classification_report(
    all_labels,
    all_preds,
    target_names=class_names_cnn3,
    zero_division=0
)
print(report)

Interface

In [ ]:
"""
=============================================================================
CRYLENS — INFANT HEALTH ASSISTANT
=============================================================================
Two modes:
  🍼 CRY ANALYSIS   — Audio → CNN0 → CNN1 → CNN2 → MedGemma
  🔬 SKIN ANALYSIS  — Image → CNN3 → MedGemma (dermatologist mode)

Tone: Knowledgeable nurse talking to first-time parents.
      Warm, clear, specific. Never vague. Never dismissive.

SETUP:
  !pip install gradio librosa matplotlib torchvision transformers accelerate nest_asyncio
=============================================================================
"""

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import time
import numpy as np
import torch
import torch.nn.functional as F
import librosa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image
import gradio as gr
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
CNN0_WEIGHTS       = "/content/cnn0_cry_detector.pth"
CNN1_WEIGHTS       = "/content/cnn1_binary.pth"
CNN2_WEIGHTS       = "/content/cnn2_subtype.pth"
CLASS_NAMES_PATH   = "/content/cnn2_class_names.npy"
CNN3_WEIGHTS       = "/content/cnn3_skin_classifier.pth"
CNN3_CLASSES_PATH  = "/content/cnn3_class_names.npy"
MEDGEMMA_ID        = "google/medgemma-1.5-4b-it"
TEMP_DIR           = "/content/temp_interface"
os.makedirs(TEMP_DIR, exist_ok=True)

gpu_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu_device = torch.device("cpu")
print(f"Device: {gpu_device} | CNNs on CPU, MedGemma on GPU")

# CNN3 skin labels (fallback if .npy not found)
CNN3_CLASSES_DEFAULT = [
    "Atopic Dermatitis Photos",
    "Cellulitis Impetigo and other Bacterial Infections",
    "Exanthems and Drug Eruptions",
    "Tinea Ringworm Candidiasis and other Fungal Infections",
    "Scabies Lyme Disease and other Infestations and Bites",
    "Warts Molluscum and other Viral Infections",
    "Urticaria Hives"
]

# ============================================================
# LOAD ALL MODELS
# ============================================================
CNN_AVAILABLE    = False
SKIN_AVAILABLE   = False
GEMMA_AVAILABLE  = False
model0 = model1 = model2 = model3 = None
class_names = class_names_skin = processor = medgemma = None

# --- Cry CNNs ---
cnn0_exists = os.path.exists(CNN0_WEIGHTS)
cnn1_exists = os.path.exists(CNN1_WEIGHTS)
cnn2_exists = os.path.exists(CNN2_WEIGHTS)
cn_exists   = os.path.exists(CLASS_NAMES_PATH)

if cnn0_exists and cnn1_exists and cnn2_exists and cn_exists:
    try:
        print("⏳ Loading CNN0 (cry detector)...")
        model0 = models.resnet18(pretrained=False)
        model0.fc = nn.Linear(model0.fc.in_features, 2)
        model0.load_state_dict(torch.load(CNN0_WEIGHTS, map_location=cpu_device))
        model0 = model0.to(cpu_device).eval()

        print("⏳ Loading CNN1 (hungry/not)...")
        model1 = models.resnet18(pretrained=False)
        model1.fc = nn.Linear(model1.fc.in_features, 2)
        model1.load_state_dict(torch.load(CNN1_WEIGHTS, map_location=cpu_device))
        model1 = model1.to(cpu_device).eval()

        print("⏳ Loading CNN2 (subtype)...")
        class_names = np.load(CLASS_NAMES_PATH, allow_pickle=True)
        model2 = models.resnet18(pretrained=False)
        model2.fc = nn.Linear(model2.fc.in_features, len(class_names))
        model2.load_state_dict(torch.load(CNN2_WEIGHTS, map_location=cpu_device))
        model2 = model2.to(cpu_device).eval()

        CNN_AVAILABLE = True
        print(f"✅ Cry CNNs loaded | Classes: {list(class_names)}")
    except Exception as e:
        print(f"⚠️  Cry CNN load failed: {e}")

# --- Skin CNN ---
if os.path.exists(CNN3_WEIGHTS):
    try:
        print("⏳ Loading CNN3 (skin classifier)...")
        if os.path.exists(CNN3_CLASSES_PATH):
            class_names_skin = np.load(CNN3_CLASSES_PATH, allow_pickle=True).tolist()
        else:
            class_names_skin = CNN3_CLASSES_DEFAULT

        model3 = models.resnet18(pretrained=False)
        model3.fc = nn.Linear(model3.fc.in_features, len(class_names_skin))
        model3.load_state_dict(torch.load(CNN3_WEIGHTS, map_location=cpu_device))
        model3 = model3.to(cpu_device).eval()

        SKIN_AVAILABLE = True
        print(f"✅ CNN3 loaded | {len(class_names_skin)} skin classes")
    except Exception as e:
        print(f"⚠️  CNN3 load failed: {e}")
else:
    print("⚠️  CNN3 weights not found → Skin mode unavailable")

# --- MedGemma ---
if CNN_AVAILABLE or SKIN_AVAILABLE:
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"   GPU free before MedGemma: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
    try:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        print("⏳ Loading MedGemma (~60s)...")
        processor = AutoProcessor.from_pretrained(MEDGEMMA_ID)
        medgemma  = AutoModelForImageTextToText.from_pretrained(
            MEDGEMMA_ID, torch_dtype=torch.bfloat16, device_map="auto"
        )
        medgemma.eval()
        GEMMA_AVAILABLE = True
        print("✅ MedGemma loaded")
    except Exception as e:
        print(f"⚠️  MedGemma skipped: {e}")

# ============================================================
# TRANSFORMS
# ============================================================
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# ============================================================
# CRY ANALYSIS HELPERS
# ============================================================
def audio_to_tensor(y, sr):
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    norm   = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    img    = Image.fromarray((norm * 255).astype(np.uint8)).convert("RGB").resize((224, 224))
    return transform_val(img).unsqueeze(0)


def detect_cry(y, sr, tensor):
    if model0 is not None:
        with torch.no_grad():
            p0 = F.softmax(model0(tensor), dim=1).numpy()[0]
        cry_prob = float(p0[1])
        detected = cry_prob > 0.5
        detail   = f"CNN0: cry={cry_prob:.3f}, non_cry={float(p0[0]):.3f}"
        return detected, cry_prob, "CNN0 (Kaggle-trained)", detail
    else:
        rms      = float(np.mean(librosa.feature.rms(y=y)))
        zcr      = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        flatness = float(np.mean(librosa.feature.spectral_flatness(y=y)))
        fails = []
        if rms < 0.01:     fails.append(f"RMS too low ({rms:.5f})")
        if zcr < 0.04:     fails.append(f"ZCR too low ({zcr:.5f})")
        if flatness > 0.3: fails.append(f"Flatness too high ({flatness:.4f})")
        detected = len(fails) == 0
        detail   = "All checks passed" if detected else " | ".join(fails)
        return detected, min(rms / 0.1, 1.0), "Acoustic Heuristic (fallback)", detail


def extract_features(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return {
        "duration_s":           round(librosa.get_duration(y=y, sr=sr), 2),
        "rms_energy":           round(float(np.mean(librosa.feature.rms(y=y))), 5),
        "zero_crossing_rate":   round(float(np.mean(librosa.feature.zero_crossing_rate(y))), 5),
        "spectral_centroid_hz": round(float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))), 1),
        "spectral_rolloff_hz":  round(float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))), 1),
        "spectral_bandwidth_hz":round(float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))), 1),
        "spectral_flatness":    round(float(np.mean(librosa.feature.spectral_flatness(y=y))), 4),
        "mfcc_means":           [round(v, 2) for v in mfccs.mean(axis=1).tolist()],
    }


def run_classification_cnns(tensor):
    with torch.no_grad():
        p1 = F.softmax(model1(tensor), dim=1).numpy()[0]
    hungry_p     = float(p1[1])
    not_hungry_p = float(p1[0])
    result = {"hungry": hungry_p, **{n: 0.0 for n in class_names}}
    if hungry_p <= 0.5:
        with torch.no_grad():
            p2 = F.softmax(model2(tensor), dim=1).numpy()[0]
        for i, name in enumerate(class_names):
            result[name] = float(p2[i]) * not_hungry_p
    return result


# ============================================================
# PLOTS — CRY MODE
# ============================================================
BG = "#0c0c1e"

def _save(fig, name):
    path = os.path.join(TEMP_DIR, name)
    plt.savefig(path, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    return path

def _style(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color="#e0e0ff", fontsize=10, fontweight="bold")
    ax.tick_params(colors="#8888aa", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#222244")

def plot_waveform(y, sr):
    t = np.linspace(0, len(y)/sr, num=len(y))
    fig, ax = plt.subplots(figsize=(8, 2.5), facecolor=BG)
    ax.plot(t, y, color="#4fc3f7", linewidth=0.5, alpha=0.9)
    ax.axhline(0, color="#ffffff", linewidth=0.2, alpha=0.2)
    ax.set_xlabel("Time (s)", color="#8888aa", fontsize=8)
    ax.set_ylabel("Amplitude", color="#8888aa", fontsize=8)
    _style(ax, "Waveform")
    plt.tight_layout(pad=0.5)
    return _save(fig, "waveform.png")

def plot_spectrogram(y, sr):
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    fig, ax = plt.subplots(figsize=(8, 3), facecolor=BG)
    img = ax.imshow(mel_db, aspect="auto", origin="lower",
                    cmap="magma", extent=[0, len(y)/sr, 0, sr/2/1000])
    cb = plt.colorbar(img, ax=ax, format="%+2.0f dB")
    cb.ax.tick_params(colors="#8888aa", labelsize=7)
    ax.set_xlabel("Time (s)", color="#8888aa", fontsize=8)
    ax.set_ylabel("Frequency (kHz)", color="#8888aa", fontsize=8)
    _style(ax, "Mel Spectrogram")
    plt.tight_layout(pad=0.5)
    return _save(fig, "spectrogram.png")

def plot_mfcc(feats):
    vals   = feats["mfcc_means"]
    labels = [f"C{i+1}" for i in range(len(vals))]
    colors = ["#4fc3f7" if v >= 0 else "#e57373" for v in vals]
    fig, ax = plt.subplots(figsize=(8, 3), facecolor=BG)
    ax.bar(labels, vals, color=colors, alpha=0.85, width=0.6)
    ax.axhline(0, color="#ffffff", linewidth=0.4, alpha=0.25)
    ax.set_xlabel("MFCC Coefficient", color="#8888aa", fontsize=8)
    ax.set_ylabel("Mean Value", color="#8888aa", fontsize=8)
    _style(ax, "MFCCs — Timbral Fingerprint")
    plt.tight_layout(pad=0.5)
    return _save(fig, "mfcc.png")

def plot_diagnostics(feats, cry_detected, cry_prob, method):
    labels = ["RMS\nEnergy", "Zero\nCrossing", "Spec\nCentroid", "Spec\nRolloff", "Flatness\n(inv)"]
    vals = [
        min(feats["rms_energy"] / 0.1, 1.0),
        min(feats["zero_crossing_rate"] / 0.3, 1.0),
        min(feats["spectral_centroid_hz"] / 8000, 1.0),
        min(feats["spectral_rolloff_hz"] / 10000, 1.0),
        max(0.0, 1.0 - feats["spectral_flatness"] / 0.5),
    ]
    thresholds = [0.1, 0.13, 0.25, 0.2, 0.4]
    colors = ["#4fc3f7" if v > t else "#e57373" for v, t in zip(vals, thresholds)]
    actual = [feats["rms_energy"], feats["zero_crossing_rate"],
              feats["spectral_centroid_hz"], feats["spectral_rolloff_hz"], feats["spectral_flatness"]]
    fig, ax = plt.subplots(figsize=(8, 3), facecolor=BG)
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, width=0.5)
    for bar, raw in zip(bars, actual):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                f"{raw:.4f}", ha="center", color="#ccddff", fontsize=7)
    ax.set_ylim(0, 1.35)
    ax.set_ylabel("Normalised Score", color="#8888aa", fontsize=8)
    verdict = f"{'✅ CRY' if cry_detected else '🔇 NOT A CRY'} — CNN0: {cry_prob:.2f} | {method}"
    _style(ax, verdict)
    plt.tight_layout(pad=0.5)
    return _save(fig, "diagnostics.png")

def plot_probabilities(probs):
    labels = list(probs.keys())
    values = [probs[k] * 100 for k in labels]
    colors = ["#e57373" if k == "hungry" else "#4fc3f7" for k in labels]
    fig, ax = plt.subplots(figsize=(7, 3), facecolor=BG)
    bars = ax.barh(labels, values, color=colors, alpha=0.85, height=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va="center", color="#e0e0ff", fontsize=8)
    ax.set_xlabel("Probability (%)", color="#8888aa", fontsize=8)
    ax.set_xlim(0, 115)
    _style(ax, "Cry Type Probabilities")
    plt.tight_layout(pad=0.5)
    return _save(fig, "probs.png")

def plot_skin_probabilities(probs_dict):
    labels = list(probs_dict.keys())
    # Shorten long class names for display
    short_labels = [l.split(" Photos")[0].split(" and other")[0][:35] for l in labels]
    values = [probs_dict[k] * 100 for k in labels]
    colors = ["#b39ddb" if v == max(values) else "#4fc3f7" for v in values]
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=BG)
    bars = ax.barh(short_labels, values, color=colors, alpha=0.85, height=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va="center", color="#e0e0ff", fontsize=8)
    ax.set_xlabel("Probability (%)", color="#8888aa", fontsize=8)
    ax.set_xlim(0, 120)
    _style(ax, "Skin Condition Probabilities (CNN3)")
    plt.tight_layout(pad=0.5)
    return _save(fig, "skin_probs.png")


# ============================================================
# MEDGEMMA — CRY MODE PROMPT
# ============================================================
def get_cry_medgemma_response(probs, feats, waveform_path):
    pred      = max(probs, key=probs.get)
    conf      = probs[pred]
    probs_str = ", ".join(f"{k}: {v:.2f}" for k, v in sorted(probs.items(), key=lambda x: -x[1]))
    mfcc_str  = ", ".join(f"C{i+1}={v}" for i, v in enumerate(feats["mfcc_means"][:5]))

    if conf >= 0.85:
        confidence_frame = f"highly confident ({conf:.0%})"
    elif conf >= 0.65:
        confidence_frame = f"moderately confident ({conf:.0%})"
    else:
        confidence_frame = f"low confidence ({conf:.0%}) — but {pred.replace('_', ' ')} remains most likely"

    prompt_text = f"""You are a warm, knowledgeable pediatric nurse helping first-time parents understand why their baby is crying. Speak directly to a parent — reassuring but specific, never dismissive, never vague.

A trained acoustic CNN has analyzed the cry and produced this diagnosis.

DIAGNOSIS: {pred.upper().replace('_', ' ')} — model is {confidence_frame}
Full CNN probability distribution: {probs_str}

Acoustic evidence:
Duration: {feats['duration_s']}s | RMS energy: {feats['rms_energy']} ({'high intensity' if feats['rms_energy'] > 0.05 else 'moderate/low intensity'}) | Spectral centroid: {feats['spectral_centroid_hz']} Hz ({'high-pitched' if feats['spectral_centroid_hz'] > 3000 else 'lower-pitched'}) | Zero crossing rate: {feats['zero_crossing_rate']} ({'irregular pattern' if feats['zero_crossing_rate'] > 0.1 else 'smooth pattern'}) | MFCCs: {mfcc_str}

STRICT RULES:
1. The diagnosis is {pred.replace('_', ' ')}. Do NOT list other cry types as possible causes.
2. Every section must be specific to {pred.replace('_', ' ')} — not generic baby advice.
3. Name actual techniques, positions, or safe OTC options specific to {pred.replace('_', ' ')}.
4. In When_to_Call_Doctor: use concrete numbers — specific temperatures, durations, behaviors.
5. Write like a nurse talking to a first-time parent — warm, clear, step-by-step.
6. Complete every section fully without cutting off.

Condition_Overview:
[Explain to a first-time parent what {pred.replace('_', ' ')} means, what's happening in their baby's body right now, and what the cry pattern tells us about how uncomfortable the baby is]

Immediate_Actions:
[Step-by-step what the parent should do in the next 5-10 minutes, specific to {pred.replace('_', ' ')}]

Treatment_and_Relief:
[Specific safe techniques, holds, or OTC options for {pred.replace('_', ' ')} — name them, describe how to do them]

Warning_Signs:
[Tell the parent exactly what changes would mean this has become more serious — be specific, not generic]

When_to_Call_Doctor:
[Concrete thresholds: exact temperatures, how many hours of crying, feeding refusal duration, specific behavioral changes that mean "call now"]

Disclaimer:
[One warm sentence — AI guidance to help you, but your pediatrician is always the final word]
"""

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
    prompt   = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    wf_img   = Image.open(waveform_path).convert("RGB")

    torch.cuda.empty_cache()
    inputs = processor(text=prompt, images=wf_img, return_tensors="pt")
    inputs = {k: v.to(medgemma.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = medgemma.generate(**inputs, max_new_tokens=700, temperature=0.3, do_sample=False)

    text = processor.decode(out[0], skip_special_tokens=True)
    return (text[text.index("Condition_Overview:"):] if "Condition_Overview:" in text else text), pred, conf


# ============================================================
# MEDGEMMA — SKIN MODE PROMPT
# ============================================================
def get_skin_medgemma_response(pred_label, conf, probs_dict, skin_image_path):
    probs_str  = ", ".join(f"{k.split(' Photos')[0].split(' and other')[0]}: {v:.2f}"
                           for k, v in sorted(probs_dict.items(), key=lambda x: -x[1]))

    if conf >= 0.85:
        confidence_frame = f"highly confident ({conf:.0%})"
    elif conf >= 0.65:
        confidence_frame = f"moderately confident ({conf:.0%})"
    else:
        confidence_frame = f"low confidence ({conf:.0%}) — {pred_label} is the closest match but the image is ambiguous"

    # Clean up the label for display
    clean_label = pred_label.replace(" Photos", "").replace(" and other", " /")

    prompt_text = f"""You are a warm, knowledgeable pediatric nurse with dermatology expertise, helping first-time parents understand a skin condition on their baby. Speak directly to a parent — calm, reassuring, specific. Never use alarming language unnecessarily. Never be vague.

A trained skin classifier (CNN) has analyzed the image of the baby's skin.

DIAGNOSIS: {clean_label} — model is {confidence_frame}
Full probability distribution: {probs_str}

The image of the baby's skin is attached for your visual assessment.

STRICT RULES:
1. The diagnosis is {clean_label}. Do NOT list other skin conditions as primary alternatives unless confidence is genuinely low.
2. Every section must be specific to {clean_label} as it typically presents in infants and young children.
3. Describe what this looks like visually so the parent can confirm it matches what they see.
4. Name actual safe topical treatments, hygiene measures, or OTC options appropriate for infants.
5. In When_to_Call_Doctor: use concrete thresholds — spreading by X cm, fever above specific temperature, duration, signs of secondary infection.
6. Tone: you are a nurse, not a lawyer. Be warm, practical, and direct. First-time parents are anxious — reassure them while being honest.
7. Complete every section fully without cutting off.

Condition_Overview:
[Explain to a first-time parent what {clean_label} is, how it typically looks and feels on a baby's skin, and whether it is common or something that warrants more concern]

What_You_Might_See:
[Describe the visual appearance — color, texture, location patterns, size — so the parent can confirm it matches their baby's skin. Reference the uploaded image.]

Immediate_Care:
[What the parent should do today — specific hygiene, clothing, environment changes, and any safe immediate relief for the baby's discomfort]

Safe_Treatments:
[Specific safe topical or OTC options appropriate for infants with {clean_label} — name them and briefly explain how to use them. Note any treatments to avoid in infants.]

Warning_Signs:
[Specific visual or behavioral changes that mean {clean_label} has become infected, spread dangerously, or is actually something more serious]

When_to_Call_Doctor:
[Concrete escalation criteria — spreading area, fever above specific temperature in °C and °F, duration without improvement, signs of secondary bacterial infection, infant distress level]

Disclaimer:
[One warm sentence — this AI assessment helps you understand what you might be seeing, but a pediatrician or dermatologist should confirm any skin diagnosis]
"""

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
    prompt   = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    skin_img = Image.open(skin_image_path).convert("RGB")

    torch.cuda.empty_cache()
    inputs = processor(text=prompt, images=skin_img, return_tensors="pt")
    inputs = {k: v.to(medgemma.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = medgemma.generate(**inputs, max_new_tokens=750, temperature=0.3, do_sample=False)

    text = processor.decode(out[0], skip_special_tokens=True)
    return text[text.index("Condition_Overview:"):] if "Condition_Overview:" in text else text


# ============================================================
# MAIN — CRY ANALYSIS
# ============================================================
def analyze_cry(audio_file):
    if audio_file is None:
        return "⚠️ Please upload an audio file.", None, None, None, None, None, "Upload audio to begin."

    t0 = time.time()
    try:
        y, sr         = librosa.load(audio_file, sr=22050)
        tensor        = audio_to_tensor(y, sr)
        waveform_path = plot_waveform(y, sr)
        spec_path     = plot_spectrogram(y, sr)
        feats         = extract_features(y, sr)
        mfcc_path     = plot_mfcc(feats)

        feats_text = (
            f"Duration          : {feats['duration_s']} s\n"
            f"RMS Energy        : {feats['rms_energy']}\n"
            f"Zero Crossing Rate: {feats['zero_crossing_rate']}\n"
            f"Spectral Centroid : {feats['spectral_centroid_hz']} Hz\n"
            f"Spectral Rolloff  : {feats['spectral_rolloff_hz']} Hz\n"
            f"Spectral Bandwidth: {feats['spectral_bandwidth_hz']} Hz\n"
            f"Spectral Flatness : {feats['spectral_flatness']}"
        )

        cry_detected, cry_prob, method, cry_detail = detect_cry(y, sr, tensor)
        diag_path = plot_diagnostics(feats, cry_detected, cry_prob, method)

        if not cry_detected:
            status = (
                f"🔇 NOT A CRY DETECTED\n\n"
                f"Detection method : {method}\n"
                f"Cry probability  : {cry_prob:.3f}\n"
                f"Detail           : {cry_detail}\n\n"
                f"--- Acoustic Features ---\n{feats_text}\n\n"
                f"⏱ {time.time()-t0:.1f}s"
            )
            return status, waveform_path, spec_path, mfcc_path, diag_path, None, \
                   "No cry detected — classification skipped."

        prob_path  = None
        gemma_text = ""
        pred_label = "unknown"
        conf       = 0.0

        if CNN_AVAILABLE:
            probs      = run_classification_cnns(tensor)
            prob_path  = plot_probabilities(probs)
            pred_label = max(probs, key=probs.get)
            conf       = probs[pred_label]

            if GEMMA_AVAILABLE:
                gemma_text, pred_label, conf = get_cry_medgemma_response(probs, feats, waveform_path)
            else:
                gemma_text = "MedGemma not loaded. CNN classification above is available."
        else:
            gemma_text = "Cry CNNs not trained yet. Run training script first."

        status = (
            f"✅ CRY DETECTED\n\n"
            f"Detection method : {method}\n"
            f"Cry probability  : {cry_prob:.3f}\n\n"
            + (f"Predicted type   : {pred_label.upper().replace('_', ' ')}\n"
               f"Confidence       : {conf:.0%}\n\n" if CNN_AVAILABLE else "")
            + f"--- Acoustic Features ---\n{feats_text}\n\n"
            f"⏱ {time.time()-t0:.1f}s"
        )

        return status, waveform_path, spec_path, mfcc_path, diag_path, prob_path, gemma_text

    except Exception as e:
        import traceback
        return f"❌ Error: {e}", None, None, None, None, None, traceback.format_exc()


# ============================================================
# MAIN — SKIN ANALYSIS
# ============================================================
def analyze_skin(image_file):
    if image_file is None:
        return "⚠️ Please upload a skin image.", None, "Upload a clear photo of the skin area to begin."

    t0 = time.time()
    try:
        if not SKIN_AVAILABLE:
            return "⚠️ CNN3 skin model not loaded.", None, "Train CNN3 first (cnn3_skin_classifier.pth)."

        skin_img = Image.open(image_file).convert("RGB")
        tensor   = transform_val(skin_img).unsqueeze(0)

        with torch.no_grad():
            p3 = F.softmax(model3(tensor), dim=1).numpy()[0]

        probs_dict = {class_names_skin[i]: float(p3[i]) for i in range(len(class_names_skin))}
        pred_label = max(probs_dict, key=probs_dict.get)
        conf       = probs_dict[pred_label]
        prob_path  = plot_skin_probabilities(probs_dict)

        clean_pred = pred_label.replace(" Photos", "").replace(" and other", " /")

        status = (
            f"🔬 SKIN ANALYSIS COMPLETE\n\n"
            f"Predicted condition : {clean_pred}\n"
            f"Confidence          : {conf:.0%}\n\n"
            f"⏱ {time.time()-t0:.1f}s"
        )

        if GEMMA_AVAILABLE:
            gemma_text = get_skin_medgemma_response(pred_label, conf, probs_dict, image_file)
        else:
            gemma_text = f"CNN3 diagnosis: {clean_pred} ({conf:.0%})\n\nMedGemma not loaded for detailed guidance."

        return status, prob_path, gemma_text

    except Exception as e:
        import traceback
        return f"❌ Error: {e}", None, traceback.format_exc()


# ============================================================
# GRADIO UI — TWO TABS
# ============================================================
CRY_GATE   = "CNN0 (Kaggle-trained)" if model0 is not None else "Heuristic (fallback)"
CRY_MODE   = "🟢 FULL" if (CNN_AVAILABLE and GEMMA_AVAILABLE) else "🟡 CNN" if CNN_AVAILABLE else "🔵 DIAGNOSTIC"
SKIN_MODE  = "🟢 FULL" if (SKIN_AVAILABLE and GEMMA_AVAILABLE) else "🟡 CNN only" if SKIN_AVAILABLE else "🔴 UNAVAILABLE"

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600&display=swap');
body, .gradio-container { background:#07071a!important; font-family:'DM Sans',sans-serif!important; }
.hdr { text-align:center; padding:2rem 1rem 1.5rem;
       background:linear-gradient(135deg,#0f0f2e,#1a1a4e,#0f1a3e);
       border-bottom:1px solid #2a2a6a; margin-bottom:1.5rem; }
.ttl { font-family:'Space Mono',monospace!important; font-size:1.9rem; font-weight:700;
       color:#e0e8ff; letter-spacing:-0.02em; margin:0 0 .4rem; }
.sub { font-size:.88rem; color:#7788bb; margin:0; }
.brow { display:flex; justify-content:center; gap:.6rem; margin-top:.8rem; flex-wrap:wrap; }
.badge { background:rgba(79,195,247,.12); border:1px solid rgba(79,195,247,.3); color:#4fc3f7;
         border-radius:20px; padding:.2rem .75rem; font-size:.7rem; font-family:'Space Mono',monospace; }
.sbadge { background:rgba(179,157,219,.12); border:1px solid rgba(179,157,219,.3); color:#b39ddb;
          border-radius:20px; padding:.2rem .75rem; font-size:.7rem; font-family:'Space Mono',monospace; }
.mbadge { background:rgba(100,200,100,.1); border:1px solid rgba(100,200,100,.3); color:#88dd88;
          border-radius:20px; padding:.2rem .75rem; font-size:.7rem; font-family:'Space Mono',monospace; }
.lbl { font-family:'Space Mono',monospace; font-size:.67rem; letter-spacing:.12em;
       color:#5566aa; text-transform:uppercase; margin-bottom:.35rem; }
button.primary { background:linear-gradient(135deg,#1a3a8f,#0e2a6e)!important;
                 border:1px solid #2a4aaf!important; color:#e0e8ff!important;
                 font-family:'Space Mono',monospace!important; font-size:.82rem!important;
                 border-radius:8px!important; padding:.7rem 1.5rem!important; }
button.primary:hover { background:linear-gradient(135deg,#2a4aaf,#1a3a9f)!important;
                       box-shadow:0 0 18px rgba(42,74,175,.45)!important; }
.disc { text-align:center; color:#445588; font-size:.7rem; padding:.8rem;
        border-top:1px solid #111138; margin-top:1rem; font-family:'Space Mono',monospace; }
"""

with gr.Blocks(css=CSS, title="FirstSteps-AI") as demo:

    gr.HTML(f"""
    <div class="hdr">
        <p class="ttl">FirstSteps-AI </p>
        <p class="sub">Infant Health Assistant · Cry Analysis + Skin Dermatology · Powered by MedGemma</p>
        <div class="brow">
            <span class="badge">CNN0: Cry Gate</span>
            <span class="badge">CNN1+2: Cry Type</span>
            <span class="sbadge">CNN3: Skin</span>
            <span class="badge">MedGemma</span>
            <span class="mbadge">Cry: {CRY_MODE}</span>
            <span class="mbadge">Skin: {SKIN_MODE}</span>
        </div>
    </div>
    """)

    with gr.Tabs():

        # ── TAB 1: CRY ANALYSIS ──────────────────────────────────
        with gr.Tab("🍼 Cry Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<p class="lbl">Upload Baby Audio</p>')
                    audio_in = gr.Audio(label="WAV / MP3 / OGG / M4A", type="filepath")
                    cry_btn  = gr.Button("🔬 Analyze Cry", variant="primary")
                    gr.HTML(f"""
                    <div style="margin-top:1rem;padding:.9rem;background:#090922;
                                border:1px solid #151545;border-radius:10px;">
                        <p style="color:#4488cc;font-family:'Space Mono',monospace;
                                  font-size:.67rem;letter-spacing:.1em;margin:0 0 .35rem;">PIPELINE</p>
                        <p style="color:#6677aa;font-size:.77rem;line-height:1.65;margin:0;">
                        1. CNN0 → Cry / Non-cry ({CRY_GATE})<br>
                        2. Waveform + Spectrogram + MFCC<br>
                        3. Acoustic diagnostic chart<br>
                        {"4. CNN1 → Hungry / Not-hungry<br>"
                         "5. CNN2 → Subtype<br>"
                         "6. MedGemma → Nurse guidance"
                         if CNN_AVAILABLE else "4–6. Train cry CNNs first"}
                        </p>
                    </div>
                    """)
                with gr.Column(scale=2):
                    gr.HTML('<p class="lbl">Result</p>')
                    cry_status = gr.Textbox(label="Status + Acoustic Features", lines=14)

            gr.HTML('<p class="lbl" style="margin-top:1rem;">Acoustic Visualisations</p>')
            with gr.Row():
                cry_waveform = gr.Image(label="Waveform",        type="filepath")
                cry_spec     = gr.Image(label="Mel Spectrogram", type="filepath")
            with gr.Row():
                cry_mfcc = gr.Image(label="MFCCs",              type="filepath")
                cry_diag = gr.Image(label="Acoustic Diagnostic", type="filepath")

            gr.HTML('<p class="lbl" style="margin-top:1rem;">Classification + Nurse Guidance</p>')
            with gr.Row():
                cry_probs  = gr.Image(label="Cry Type Probabilities", type="filepath")
                cry_gemma  = gr.Textbox(label="MedGemma Nurse Guidance", lines=18)

            cry_btn.click(
                fn=analyze_cry,
                inputs=[audio_in],
                outputs=[cry_status, cry_waveform, cry_spec, cry_mfcc, cry_diag, cry_probs, cry_gemma]
            )

        # ── TAB 2: SKIN ANALYSIS ─────────────────────────────────
        with gr.Tab("🔬 Skin Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<p class="lbl">Upload Skin Photo</p>')
                    skin_in  = gr.Image(label="Clear photo of skin area", type="filepath")
                    skin_btn = gr.Button("🔬 Analyze Skin", variant="primary")
                    gr.HTML(f"""
                    <div style="margin-top:1rem;padding:.9rem;background:#090922;
                                border:1px solid #151545;border-radius:10px;">
                        <p style="color:#8866cc;font-family:'Space Mono',monospace;
                                  font-size:.67rem;letter-spacing:.1em;margin:0 0 .35rem;">SKIN PIPELINE</p>
                        <p style="color:#6677aa;font-size:.77rem;line-height:1.65;margin:0;">
                        1. CNN3 → Classify skin condition<br>
                        {"2. MedGemma → Dermatologist guidance<br>"
                         "   Tone: nurse to first-time parent"
                         if SKIN_AVAILABLE else "Train CNN3 first"}
                        </p>
                        <p style="color:#445577;font-size:.7rem;margin:.6rem 0 0;font-family:'Space Mono',monospace;">
                        CONDITIONS COVERED:<br>
                        Atopic Dermatitis · Cellulitis · Impetigo<br>
                        Exanthems · Tinea · Ringworm · Candidiasis<br>
                        Scabies · Warts · Molluscum · Urticaria
                        </p>
                    </div>
                    """)
                with gr.Column(scale=2):
                    gr.HTML('<p class="lbl">Result</p>')
                    skin_status = gr.Textbox(label="CNN3 Skin Classification", lines=6)

            gr.HTML('<p class="lbl" style="margin-top:1rem;">Condition Probabilities + Dermatologist Guidance</p>')
            with gr.Row():
                skin_probs = gr.Image(label="Skin Condition Probabilities", type="filepath")
                skin_gemma = gr.Textbox(label="MedGemma Dermatologist Guidance", lines=22)

            skin_btn.click(
                fn=analyze_skin,
                inputs=[skin_in],
                outputs=[skin_status, skin_probs, skin_gemma]
            )

    gr.HTML("""
    <div class="disc">
        ⚠️ FirstSteps-AI is a research prototype — not a substitute for professional medical advice.<br>
        Always consult a qualified pediatrician or dermatologist for diagnosis and treatment decisions.
    </div>
    """)

import nest_asyncio
nest_asyncio.apply()

demo.launch(
    share=True,
    debug=False,
    server_name="0.0.0.0",
    server_port=7880,
    allowed_paths=[TEMP_DIR],
    prevent_thread_lock=True
)


**Resources:**
- [Model on HuggingFace](https://huggingface.co/google/medgemma-1.5-4b-it)
- [Official GitHub](https://github.com/Google-Health/medgemma)
- [Model Card & Documentation](https://developers.google.com/health-ai-developer-foundations/medgemma)
- [Research Blog Post](https://research.google/blog/next-generation-medical-image-interpretation-with-medgemma-15-and-medical-speech-to-text-with-medasr/)

---

## Citation

```bibtex
@article{sellergren2025medgemma,
  title={MedGemma Technical Report},
  author={Sellergren et al.},
  journal={arXiv preprint arXiv:2507.05201},
  year={2025}
}
```

---